# Index Table Preparation

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# config
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

Pandas notes for me: 

gdf.loc[ROWS, COLUMNS] = "select rows by this value, and columns by this value"

new field:
gdf["new_field"] = (
    gdf["field_a"].astype(str)
    + "_"
    + gdf["field_b"].astype(str)
    + "_"
    + gdf["field_c"].astype(str)
)

### Load the Shapefile and give it a true unique ID before doing anything else to it. 

In [ ]:
# this is the path to the original feature service, downloaded as a shapefile from here: 
# https://www.arcgis.com/home/item.html?id=f260e4065a384e7ebee05b23362f380f#overview
# Layer 36112 - https://gis5.uspatial.umn.edu/arcgis/rest/services/mhapo/MHAPO_dynamic/MapServer/0
original_index_path = "data/historic_airphoto_index/shapefiles/umn_airphoto_index_original.shp"

gdf_original = gpd.read_file(original_index_path)

print(gdf_original.crs)
# we're keeping this in web mercator for now. 

In [ ]:
# make a unique identifier

gdf_original["source_id"] = range(1, len(gdf_original) + 1)

assert gdf_original["source_id"].is_unique
assert gdf_original["source_id"].notna().all()

In [ ]:
gdf_original

In [ ]:
gdf_original.to_file(
    "data/historic_airphoto_index/umn_index_original_with_source_id.gpkg",
    driver="GPKG"
)

### Cleaning the original table

In [ ]:
gdf_clean = gdf_original.copy()

#preview the columns
gdf_clean.columns

#### Cleaning the URLs
And looking for true duplicates...

In [ ]:
# clean URL spaces

gdf_clean["url_link"] = gdf_clean["url_link"].str.strip()

In [ ]:
exists = 'https://geo.lib.umn.edu/Olmsted/1940/WB-3A-115.jpg' in gdf_clean['url_link'].values

exists

In [ ]:
# identify duplicate URLs

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [ ]:
# identify duplicate URLs with unique photo_ids to be fixed

fake_duplicates = gdf_clean[
    gdf_clean["url_link"].duplicated(keep=False)
    & ~gdf_clean["photo_id"].duplicated(keep=False)]
fake_duplicates = fake_duplicates.sort_values(by=['flight_cod','url_link'])
fake_duplicates

In [ ]:
# fix olmsted county URLs

bad_url = "https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg"

mask = gdf_clean["url_link"] == bad_url

gdf_clean.loc[mask, "url_link"] = (
    "https://geo.lib.umn.edu/Olmsted/1940/"
    + gdf_clean.loc[mask, "photo_id"]
    + ".jpg"
)

In [ ]:
check = gdf_clean[gdf_clean["flight_cod"] == "Olmsted"]

check

In [ ]:
# identify duplicate URLs again

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [ ]:
# drop duplicates arbitrarily (the only thing that varies is the geometry and we're not using it.)
gdf_clean = gdf_clean.drop_duplicates(
    subset=["url_link"],
    keep="first"
).copy()

In [ ]:
# identify duplicate URLs again
# there should be none. 

duplicates = gdf_clean[gdf_clean["photo_id"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','photo_id'])
duplicates

In [ ]:
print("Unique URLs:", gdf_clean["url_link"].is_unique)
print("Remaining records:", len(gdf_clean))

### Clean up other values

In [ ]:
# convert year field to an integer rather than a float
# handles missing values
gdf_clean["year_"] = gdf_clean["year_"].astype("Int64")

In [ ]:
# populate missing counties from the flight code field
gdf_clean["name10"] = gdf_clean["name10"].fillna(
    gdf_clean["flight_cod"]
)

In [ ]:
# check for nulls
print(gdf_clean["name10"].isna().any())

print(gdf_clean["year_"].isna().any())

print(gdf_clean["photo_id"].isna().any())

print(gdf_clean["url_link"].isna().any())

### Adding filename identifier and validation

In [ ]:
# add a new unique, human readable identifier field called "cty_year_id"
# it will be used for naming files
# it is a combination of the county, the year, and the photo ID. 

gdf_clean["cty_year_id"] = (
    gdf_clean["name10"].str.strip().str.lower().str.replace(" ", "_").str.replace('.', '', regex=False)
    + "_"
    + gdf_clean["year_"].astype("string")
    + "_"
    + gdf_clean["photo_id"].str.strip().str.lower().str.replace(" ", "_")
)

# add a number to any non-unique values produced by this
counts = gdf_clean.groupby("cty_year_id").cumcount()

gdf_clean["cty_year_id"] += counts.map(
    lambda n: f"_{n}" if n > 0 else ""
)

In [ ]:
# validate that the periods were handled correctly
gdf_clean_filtered = gdf_clean.loc[gdf_clean["name10"] == "St. Louis"]

gdf_clean_filtered

In [ ]:
# validate that this new field is unique and contains no nulls
# should return 

print(f"All unique values:", gdf_clean['cty_year_id'].is_unique)
print(f"Has null values:", gdf_clean['cty_year_id'].isna().any())
print(f"Has spaces:", gdf_clean['cty_year_id'].str.contains(' ').any())

In [ ]:
# check it out
gdf_clean

In [ ]:
# save it
gdf_clean.to_file("data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg")

### Reopen for final modifications

In [ ]:
# open cleaned table
path = "data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg"

gdf = gpd.read_file(path)

In [ ]:
gdf

### Add calculated latitude column

In [ ]:
gdf["latitude"] = gdf.to_crs("EPSG:4326").geometry.y

### Drop and reorder columns

In [ ]:
# inspect
gdf.columns

In [ ]:
# rename
new_gdf = gdf.rename(columns={
    "url_link" : "url",
    "year_" : "year",
    "name10" : "county",
    'flight_cod' : 'flight_code'
})

In [ ]:
# add
new_gdf["worldfile_json"] = None
new_gdf["h_extent"] = None
new_gdf["scale"] = None
new_gdf["dpi_x"] = None
new_gdf["width_px"] = None
new_gdf["status"] = None
new_gdf["gcp_count"] = None
new_gdf["confidence"] = None
new_gdf["date_download"] = None
new_gdf["date_georef"] = None
new_gdf["ground_width"] = None

In [ ]:
# reorder and drop
new_gdf = new_gdf[['cty_year_id','status','county','year','photo_id','h_extent','ground_width','gcp_count','confidence','scale','dpi_x','width_px','worldfile_json','url','flight_code','source_id','date_download','date_georef','latitude','geometry']]

new_gdf

In [ ]:
new_gdf = new_gdf.sort_values(by=['county','year'], ascending=[True, True])

new_gdf

### Export

In [ ]:
new_gdf.to_file("data/historic_airphoto_index/main_index_blank.gpkg")

### Edit Fields Later

In [3]:
edited_gdf = gpd.read_file("data/historic_airphoto_index/main_index_blank.gpkg")

In [8]:
# add columns
edited_gdf["ground_width"] = None


In [5]:
edited_gdf["latitude"] = edited_gdf.to_crs("EPSG:4326").geometry.y

In [6]:
edited_gdf

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_download,date_georef,geometry,ground_extent,latitude
0,aitkin_1939_bxv02017,None,Aitkin,1939,bxv02017,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxv/y1939/bxv02017.jpg,bxv,17233,None,None,POINT (-10383436.935 5806733.399),None,46.164398
1,aitkin_1939_bxt03191,None,Aitkin,1939,bxt03191,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03191.jpg,bxt,17326,None,None,POINT (-10442747.814 5821533.312),None,46.256401
2,aitkin_1939_bxt03192,None,Aitkin,1939,bxt03192,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03192.jpg,bxt,17327,None,None,POINT (-10440599.092 5821517.341),None,46.256302
3,aitkin_1939_byd03058,None,Aitkin,1939,byd03058,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd03058.jpg,byd,17348,None,None,POINT (-10359703.23 5813165.04),None,46.204399
4,aitkin_1939_byd04003,None,Aitkin,1939,byd04003,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd04003.jpg,byd,17350,None,None,POINT (-10358746.071 5818024.084),None,46.234600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115572,yellow_medicine_1991_napp_4112_148,None,Yellow Medicine,1991,napp_4112_148,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_148.jpg,napp,9647,None,None,POINT (-10655011.753 5597325.517),None,44.846100
115573,yellow_medicine_1991_napp_4112_149,None,Yellow Medicine,1991,napp_4112_149,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_149.jpg,napp,9648,None,None,POINT (-10655123.006 5602256.72),None,44.877499
115574,yellow_medicine_1991_napp_4113_036,None,Yellow Medicine,1991,napp_4113_036,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_036.jpg,napp,9649,None,None,POINT (-10648610.57 5601707.206),None,44.874000
115575,yellow_medicine_1991_napp_4113_037,None,Yellow Medicine,1991,napp_4113_037,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_037.jpg,napp,9650,None,None,POINT (-10648410.986 5596854.756),None,44.843101


In [9]:
edited_gdf = edited_gdf[['cty_year_id','status','county','year','photo_id','h_extent','ground_width','gcp_count','confidence','scale','dpi_x','width_px','worldfile_json','url','flight_code','source_id','date_download','date_georef','latitude','geometry']]


In [11]:
edited_gdf

,cty_year_id,status,county,year,photo_id,h_extent,ground_width,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_download,date_georef,latitude,geometry
0,aitkin_1939_bxv02017,None,Aitkin,1939,bxv02017,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxv/y1939/bxv02017.jpg,bxv,17233,None,None,46.164398,POINT (-10383436.935 5806733.399)
1,aitkin_1939_bxt03191,None,Aitkin,1939,bxt03191,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03191.jpg,bxt,17326,None,None,46.256401,POINT (-10442747.814 5821533.312)
2,aitkin_1939_bxt03192,None,Aitkin,1939,bxt03192,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03192.jpg,bxt,17327,None,None,46.256302,POINT (-10440599.092 5821517.341)
3,aitkin_1939_byd03058,None,Aitkin,1939,byd03058,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd03058.jpg,byd,17348,None,None,46.204399,POINT (-10359703.23 5813165.04)
4,aitkin_1939_byd04003,None,Aitkin,1939,byd04003,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd04003.jpg,byd,17350,None,None,46.234600,POINT (-10358746.071 5818024.084)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115572,yellow_medicine_1991_napp_4112_148,None,Yellow Medicine,1991,napp_4112_148,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_148.jpg,napp,9647,None,None,44.846100,POINT (-10655011.753 5597325.517)
115573,yellow_medicine_1991_napp_4112_149,None,Yellow Medicine,1991,napp_4112_149,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_149.jpg,napp,9648,None,None,44.877499,POINT (-10655123.006 5602256.72)
115574,yellow_medicine_1991_napp_4113_036,None,Yellow Medicine,1991,napp_4113_036,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_036.jpg,napp,9649,None,None,44.874000,POINT (-10648610.57 5601707.206)
115575,yellow_medicine_1991_napp_4113_037,None,Yellow Medicine,1991,napp_4113_037,None,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_037.jpg,napp,9650,None,None,44.843101,POINT (-10648410.986 5596854.756)


In [12]:
edited_gdf.to_file("data/historic_airphoto_index/main_index_blank.gpkg")